
# 06. CortexODE + ScaleSurfer

This notebook uses the **released pretrained CortexODE deformation models** in
`CortexODE/ckpts/pretrained` and plugs in our WM segmentation source instead
of retraining CortexODE from scratch.

Pipeline:

1. choose a subject from the standard FS8 `tran/val/test` split
2. get a native-space `aparc+aseg` volume
   - `APARC_SOURCE = "pred"`: predict with our `TransUNet3D` checkpoint, cache it, and resample back to `orig.mgz`
   - `APARC_SOURCE = "true"`: use FreeSurfer ground truth for oracle debugging
3. recombine that `aparc+aseg` into `lh.white` / `rh.white` masks
4. build topology-corrected white seeds locally
5. run the authors' pretrained white model, then white-to-pial handoff, then pretrained pial model
6. save `?h.white` and `?h.pial` predictions and build a FreeView command

Notes:

- The pretrained release **is available locally** in `CortexODE/ckpts/pretrained/{adni,hcp,dhcp}`.
- The released checkpoints are hemisphere-specific, so pretrained inference still loads separate `lh` and `rh` models.
- Cached predicted `aparc+aseg` volumes are written under `notebooks/cortexode_pretrained_cache`, so reruns do **not** need to re-run segmentation unless you delete the cache.


In [1]:

from pathlib import Path
import shutil
import subprocess

import nibabel as nib
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from scalesurfer.config import DEVICE, SEED, MODULE_PATH
from scalesurfer.data import build_label_lut, default_aparc_aseg_label_values
from scalesurfer.experiments import build_v8_split_from_root
from scalesurfer.metrics import dense_labels_to_fs_ids, predict_volume_from_unpadded
from scalesurfer.volume.model import TransUNet3D
from scalesurfer.surface.cortex_ode import (
    DEFAULT_PRETRAINED_ROOT,
    PretrainedCortexODEConfig,
    load_pretrained_model_bundles,
    predict_surfaces_from_native_aparc,
    resample_tensor_labels_to_native,
)

from scalesurfer.surface.metrics import evaluate_surfaces
from scalesurfer.volume.targets import load_tensor
from scalesurfer.data import save_surfaces_to_subject_dir
from scalesurfer.freeview import build_freeview_command


In [ ]:

REPO_ROOT = MODULE_PATH.parent
TENSORS_ROOT_V8 = REPO_ROOT / "data" / "tensors_gcloud"
FS8_SUBJECTS_ROOT = REPO_ROOT / "data" / "fs8" / "subjects" / "freesurfer_test_set_subjects_throughput"
SEG_CKPT_PATH = REPO_ROOT / "docs" / "notebooks" / "checkpoints_fsv8" / "fsv8_20260402_034229" / "transunet3d_best.pt"
TARGET_CACHE_ROOT = REPO_ROOT / "docs" / "notebooks" / "target_cache"
PRETRAINED_CACHE_ROOT = REPO_ROOT / "docs" / "notebooks" / "cortexode_pretrained_cache"
PRED_SURFACE_ROOT = Path("/tmp/cortexode_pretrained")

EXAMPLE_SPLIT = "test"
EXAMPLE_IDX = 0
APARC_SOURCE = "pred"   # "pred" or "true"
PRETRAINED_DATA_NAME = "adni"
AUTO_LAUNCH_FREEVIEW = False
SHOW_BOOTSTRAP_WHITE = True

split_v8 = build_v8_split_from_root(
    tensors_root=TENSORS_ROOT_V8,
    seed=int(SEED),
    ratios=(0.8, 0.1, 0.1),
)

device = torch.device(DEVICE)
config = PretrainedCortexODEConfig(
    data_name=PRETRAINED_DATA_NAME,
    device=device,
)

print(f"device={device}")
print(f"split={EXAMPLE_SPLIT} idx={EXAMPLE_IDX}")
print(f"aparc_source={APARC_SOURCE}")
print(f"pretrained_root={DEFAULT_PRETRAINED_ROOT / PRETRAINED_DATA_NAME}")


device=cuda
split=test idx=0
aparc_source=pred
pretrained_root=/home/rph/scalesurfer/scalesurfer/surface/CortexODE/ckpts/pretrained/adni


In [3]:
Path(split_v8[f"x_{EXAMPLE_SPLIT}"][EXAMPLE_IDX]).resolve()

PosixPath('/home/rph/convolutional_ar/segmentation/data/tensors_gcloud/ds002168__sub-1516__ses-none__f0a17778c3e2/mri/rawavg.pt')

In [4]:
x_path = Path(split_v8[f"x_{EXAMPLE_SPLIT}"][EXAMPLE_IDX]).resolve()
y_path = Path(split_v8[f"y_{EXAMPLE_SPLIT}"][EXAMPLE_IDX]).resolve()
subject_id = x_path.parents[1].name
subject_dir = FS8_SUBJECTS_ROOT / subject_id
metadata_path = TARGET_CACHE_ROOT / subject_id / "metadata.json"

if not subject_dir.exists():
    raise FileNotFoundError(f"Missing subject_dir: {subject_dir}")
if not metadata_path.exists():
    raise FileNotFoundError(f"Missing metadata_path: {metadata_path}")

example_row = {
    "subject_id": subject_id,
    "subject_dir": subject_dir,
    "metadata_path": metadata_path,
    "rawavg_pt": x_path,
    "aparc_aseg_pt": y_path,
}

print(f"subject_id={subject_id}")
print(f"subject_dir={subject_dir}")
print(f"rawavg_pt={x_path}")


subject_id=ds002168__sub-1516__ses-none__f0a17778c3e2
subject_dir=/home/rph/scalesurfer/data/fs8/subjects/freesurfer_test_set_subjects_throughput/ds002168__sub-1516__ses-none__f0a17778c3e2
rawavg_pt=/home/rph/convolutional_ar/segmentation/data/tensors_gcloud/ds002168__sub-1516__ses-none__f0a17778c3e2/mri/rawavg.pt


In [6]:
_seg_model = None
_class_values = None

def get_segmentation_model():
    global _seg_model
    if _seg_model is None:
        model = TransUNet3D(**dict(
            n_classes=118,
            in_channels=1,
            base_shape=(208, 240, 192),
            patch_size=(16, 16, 16),
            channels=(12, 20, 32, 48, 64, 96),
            transformer_depth=2,
            n_heads=4,
            dropout=0.0,
            positional_encoding="sincos",
        ))
        ckpt = torch.load(SEG_CKPT_PATH, map_location=device)
        model.load_state_dict(ckpt["model_state"])
        model.to(device)
        model.eval()
        _seg_model = model
    return _seg_model


def get_class_values():
    global _class_values
    if _class_values is None:
        _class_values, _ = build_label_lut(default_aparc_aseg_label_values())
    return _class_values


def predict_aparc_fs_tensor(rawavg_pt):
    raw_tensor = load_tensor(rawavg_pt).to(torch.float32)
    model = get_segmentation_model()
    dense_pred = predict_volume_from_unpadded(
        model=model,
        x_3d=raw_tensor,
        patch_size=(16, 16, 16),
        device=device,
    )
    fs_pred = dense_labels_to_fs_ids(dense_pred, class_values=get_class_values())
    if torch.is_tensor(fs_pred):
        fs_pred = fs_pred.detach().cpu().numpy()
    return np.asarray(fs_pred, dtype=np.int32)


def load_or_build_native_aparc(row, *, source):
    subject_dir = Path(row["subject_dir"])
    if source == "true":
        native_path = subject_dir / "mri" / "aparc+aseg.mgz"
        native_arr = np.asarray(nib.load(str(native_path)).get_fdata(), dtype=np.int32)
        return native_arr, native_path, "ground_truth"

    if source != "pred":
        raise ValueError(f"source must be 'pred' or 'true', got {source!r}")

    cache_dir = PRETRAINED_CACHE_ROOT / str(row["subject_id"])
    cache_dir.mkdir(parents=True, exist_ok=True)
    pred_tensor_path = cache_dir / "aparc_pred_tensor.pt"
    pred_native_path = cache_dir / "aparc_pred_native.mgz"

    if pred_native_path.exists():
        native_arr = np.asarray(nib.load(str(pred_native_path)).get_fdata(), dtype=np.int32)
        return native_arr, pred_native_path, "cached_native_prediction"

    if pred_tensor_path.exists():
        pred_tensor = torch.load(pred_tensor_path, map_location="cpu")
        pred_fs = pred_tensor.detach().cpu().numpy() if torch.is_tensor(pred_tensor) else np.asarray(pred_tensor)
    else:
        pred_fs = predict_aparc_fs_tensor(row["rawavg_pt"])
        torch.save(torch.from_numpy(np.asarray(pred_fs, dtype=np.int16)), pred_tensor_path)

    native_img = resample_tensor_labels_to_native(
        pred_fs,
        metadata_path=row["metadata_path"],
        native_ref_path=Path(row["subject_dir"]) / "mri" / "orig.mgz",
    )
    nib.save(native_img, str(pred_native_path))
    native_arr = np.asarray(native_img.get_fdata(), dtype=np.int32)
    return native_arr, pred_native_path, "fresh_native_prediction"


In [7]:
native_aparc, native_aparc_path, native_aparc_status = load_or_build_native_aparc(
    example_row,
    source=APARC_SOURCE,
)

print(f"native_aparc_status={native_aparc_status}")
print(f"native_aparc_path={native_aparc_path}")
print(f"native_aparc_shape={native_aparc.shape}")
print(f"native_aparc_unique_labels={len(np.unique(native_aparc))}")

native_aparc_status=cached_native_prediction
native_aparc_path=/home/rph/scalesurfer/docs/notebooks/cortexode_pretrained_cache/ds002168__sub-1516__ses-none__f0a17778c3e2/aparc_pred_native.mgz
native_aparc_shape=(256, 256, 256)
native_aparc_unique_labels=84


In [ ]:

model_bundles = load_pretrained_model_bundles(config=config)
result = predict_surfaces_from_native_aparc(
    subject_dir=subject_dir,
    native_aparc=native_aparc,
    config=config,
    model_bundles=model_bundles,
)

pred_surfaces = result["surfaces"]
seed_surfaces = result["seed_surfaces"]

print(f"processed_volume_shape={result['volume_shape']}")
print("predicted surfaces:")
for name in ("lh.white", "rh.white", "lh.pial", "rh.pial"):
    surf = pred_surfaces[name]
    print(f"  {name}: verts={surf['vertices_ras'].shape[0]} faces={surf['faces'].shape[0]}")


processed_volume_shape=(176, 208, 176)
predicted surfaces:
  lh.white: verts=29539 faces=59366
  rh.white: verts=30453 faces=60934
  lh.pial: verts=29539 faces=59366
  rh.pial: verts=30453 faces=60934


In [15]:
subject_dir

PosixPath('/home/rph/scalesurfer/data/fs8/subjects/freesurfer_test_set_subjects_throughput/ds002168__sub-1516__ses-none__f0a17778c3e2')

In [17]:
native_aparc.shape

(256, 256, 256)

In [12]:

pred_eval_df = evaluate_surfaces(pred_surfaces, subject_dir)
print("Pretrained CortexODE metrics (mm):")
display(pred_eval_df)

white_seed_eval_rows = []
for name in ("lh.white", "rh.white"):
    surf = seed_surfaces[name]
    ref_path = subject_dir / "surf" / name
    ref_verts, _ = nib.freesurfer.io.read_geometry(str(ref_path))
    from scalesurfer.surface.metrics import symmetric_mean_distance
    white_seed_eval_rows.append({
        "surface": name,
        "seed_sym_mean_mm": symmetric_mean_distance(surf["vertices_ras"], ref_verts),
    })
seed_eval_df = pd.DataFrame(white_seed_eval_rows)
print("Bootstrap white-seed distances (mm):")
display(seed_eval_df.round(3))


Pretrained CortexODE metrics (mm):


,surface,n_pred,n_ref,sym_mean_mm
0,lh.white,29539,135903,2.054209
1,rh.white,30453,136659,1.618087
2,lh.pial,29539,135903,1.735067
3,rh.pial,30453,136659,1.512018


Bootstrap white-seed distances (mm):


,surface,seed_sym_mean_mm
0,lh.white,2.486
1,rh.white,1.885


In [13]:

out_root = PRED_SURFACE_ROOT / str(subject_id) / APARC_SOURCE
pred_dir = out_root / "pred"
seed_dir = out_root / "seed"
pred_paths = save_surfaces_to_subject_dir(pred_surfaces, pred_dir)
seed_paths = save_surfaces_to_subject_dir(seed_surfaces, seed_dir)

cmd = build_freeview_command(
    subject_dir,
    pred_surfaces=pred_paths,
    show_orig=True,
    show_aparc=True,
    pred_edge_color="yellow",
)
if SHOW_BOOTSTRAP_WHITE:
    for name in ("lh.white", "rh.white"):
        path = seed_paths.get(name)
        if path is not None and Path(path).exists():
            cmd.append(f"{path}:edgecolor=orange:edgethickness=1")

print(f"Subject: {subject_id}")
print(f"Aparc source: {APARC_SOURCE} ({native_aparc_status})")
print("Legend: GT white=red, GT pial=cyan, pretrained CortexODE=yellow, bootstrap white seed=orange")
print("Predicted surfaces:")
for name in ("lh.white", "rh.white", "lh.pial", "rh.pial"):
    print(f"  {name}: {pred_paths.get(name)}")
print("Bootstrap white seeds:")
for name in ("lh.white", "rh.white"):
    print(f"  {name}: {seed_paths.get(name)}")
print("freeview command:")
print(" ".join(str(part) for part in cmd))

if AUTO_LAUNCH_FREEVIEW and shutil.which("freeview") is not None:
    subprocess.Popen(cmd)
elif AUTO_LAUNCH_FREEVIEW:
    print("freeview is not available in this kernel.")


Subject: ds002168__sub-1516__ses-none__f0a17778c3e2
Aparc source: pred (cached_native_prediction)
Legend: GT white=red, GT pial=cyan, pretrained CortexODE=yellow, bootstrap white seed=orange
Predicted surfaces:
  lh.white: /tmp/cortexode_pretrained/ds002168__sub-1516__ses-none__f0a17778c3e2/pred/pred/lh.white
  rh.white: /tmp/cortexode_pretrained/ds002168__sub-1516__ses-none__f0a17778c3e2/pred/pred/rh.white
  lh.pial: /tmp/cortexode_pretrained/ds002168__sub-1516__ses-none__f0a17778c3e2/pred/pred/lh.pial
  rh.pial: /tmp/cortexode_pretrained/ds002168__sub-1516__ses-none__f0a17778c3e2/pred/pred/rh.pial
Bootstrap white seeds:
  lh.white: /tmp/cortexode_pretrained/ds002168__sub-1516__ses-none__f0a17778c3e2/pred/seed/lh.white
  rh.white: /tmp/cortexode_pretrained/ds002168__sub-1516__ses-none__f0a17778c3e2/pred/seed/rh.white
freeview command:
freeview -v /home/rph/scalesurfer/data/fs8/subjects/freesurfer_test_set_subjects_throughput/ds002168__sub-1516__ses-none__f0a17778c3e2/mri/orig.mgz /hom

In [14]:
! freeview -v /home/rph/scalesurfer/data/fs8/subjects/freesurfer_test_set_subjects_throughput/ds002168__sub-1516__ses-none__f0a17778c3e2/mri/orig.mgz /home/rph/scalesurfer/data/fs8/subjects/freesurfer_test_set_subjects_throughput/ds002168__sub-1516__ses-none__f0a17778c3e2/mri/aparc+aseg.mgz -f /home/rph/scalesurfer/data/fs8/subjects/freesurfer_test_set_subjects_throughput/ds002168__sub-1516__ses-none__f0a17778c3e2/surf/lh.white:edgecolor=red:edgethickness=1:overlay=/home/rph/scalesurfer/data/fs8/subjects/freesurfer_test_set_subjects_throughput/ds002168__sub-1516__ses-none__f0a17778c3e2/surf/lh.curv /home/rph/scalesurfer/data/fs8/subjects/freesurfer_test_set_subjects_throughput/ds002168__sub-1516__ses-none__f0a17778c3e2/surf/rh.white:edgecolor=red:edgethickness=1:overlay=/home/rph/scalesurfer/data/fs8/subjects/freesurfer_test_set_subjects_throughput/ds002168__sub-1516__ses-none__f0a17778c3e2/surf/rh.curv /home/rph/scalesurfer/data/fs8/subjects/freesurfer_test_set_subjects_throughput/ds002168__sub-1516__ses-none__f0a17778c3e2/surf/lh.pial:edgecolor=cyan:edgethickness=1:overlay=/home/rph/scalesurfer/data/fs8/subjects/freesurfer_test_set_subjects_throughput/ds002168__sub-1516__ses-none__f0a17778c3e2/surf/lh.curv /home/rph/scalesurfer/data/fs8/subjects/freesurfer_test_set_subjects_throughput/ds002168__sub-1516__ses-none__f0a17778c3e2/surf/rh.pial:edgecolor=cyan:edgethickness=1:overlay=/home/rph/scalesurfer/data/fs8/subjects/freesurfer_test_set_subjects_throughput/ds002168__sub-1516__ses-none__f0a17778c3e2/surf/rh.curv /tmp/cortexode_pretrained/ds002168__sub-1516__ses-none__f0a17778c3e2/pred/pred/lh.white:edgecolor=yellow:edgethickness=2 /tmp/cortexode_pretrained/ds002168__sub-1516__ses-none__f0a17778c3e2/pred/pred/lh.pial:edgecolor=yellow:edgethickness=2 /tmp/cortexode_pretrained/ds002168__sub-1516__ses-none__f0a17778c3e2/pred/pred/rh.white:edgecolor=yellow:edgethickness=2 /tmp/cortexode_pretrained/ds002168__sub-1516__ses-none__f0a17778c3e2/pred/pred/rh.pial:edgecolor=yellow:edgethickness=2 /tmp/cortexode_pretrained/ds002168__sub-1516__ses-none__f0a17778c3e2/pred/seed/lh.white:edgecolor=orange:edgethickness=1 /tmp/cortexode_pretrained/ds002168__sub-1516__ses-none__f0a17778c3e2/pred/seed/rh.white:edgecolor=orange:edgethickness=1

Did not find any volume info
Did not find any volume info
Did not find any volume info
Did not find any volume info
